# Differential Privacy & DP-SGD
## HTB Academy — AI Privacy (Sections 7–11)

This notebook walks through:
1. **Formal definition** of (ε,δ)-differential privacy
2. **How DP-SGD works** — gradient clipping, noise addition, privacy composition
3. **Opacus setup** — per-sample gradients, architecture constraints, Poisson subsampling
4. **Training three models** — baseline, DP ε=10, DP ε=3
5. **Measuring MIA vulnerability** — confidence-threshold attack
6. **Analyzing the privacy-utility tradeoff**

---
## 0. Install Dependencies

In [ ]:
# Run once — installs Opacus and the HTB helper library
%pip install opacus safetensors
%pip install --upgrade git+https://github.com/PandaSt0rm/htb-ai-library

---
## 1. Differential Privacy — Formal Definition

A **randomized algorithm** $\mathcal{M}$ satisfies $(\varepsilon, \delta)$-differential privacy if, for any two datasets $D$ and $D'$ that differ in exactly one record, and for any subset of outputs $S$:

$$\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S] + \delta$$

| Parameter | Role | Typical value (CIFAR-10) |
|-----------|------|--------------------------|
| $\varepsilon$ | Privacy budget — **smaller = stronger privacy** | 1 – 10 |
| $\delta$ | Failure probability — must satisfy $\delta < 1/n$ | $10^{-5}$ (for $n=50{,}000$) |

**Intuition.** An attacker who observes the trained model cannot reliably tell whether Alice's record was in the training set, because the model's behavior is nearly identical with or without her data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualise the epsilon multiplicative bound: e^epsilon
epsilons = np.linspace(0.1, 10, 200)
multipliers = np.exp(epsilons)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epsilons, multipliers, lw=2)
for eps in [1, 3, 10]:
    ax.axvline(eps, ls='--', alpha=0.6, label=f'ε={eps}  →  e^ε≈{np.exp(eps):.1f}x')
ax.set_xlabel('ε  (privacy budget)')
ax.set_ylabel('e^ε  (max output-distribution ratio)')
ax.set_title('How ε controls the output-distribution bound')
ax.legend()
ax.set_ylim(0, 25)
plt.tight_layout()
plt.show()

print("Interpretation:")
for eps in [1, 3, 10]:
    print(f"  ε={eps:2d}  →  any output is at most {np.exp(eps):.1f}x more likely with your data than without")

---
## 2. How DP-SGD Works

Standard SGD: sample batch → compute average gradient → update parameters.  
DP-SGD adds three mechanisms on top:

```
For each training step:
  1. Compute per-sample gradients  g_i  for every sample i in the batch
  2. CLIP:   g_i  ←  g_i / max(1, ‖g_i‖₂ / C)      # C = max_grad_norm
  3. SUM:    G    ←  Σ g_i
  4. NOISE:  G    ←  G + N(0, σ²C²I)                # σ = noise_multiplier
  5. UPDATE: θ    ←  θ - lr · G / batch_size
```

### Why clipping first?
Clipping bounds the **L2 sensitivity** of the gradient sum to exactly $C$.  
Without this bound, we cannot calibrate the noise magnitude.

### Gaussian Mechanism
For a function with L2 sensitivity $\Delta f$, adding Gaussian noise with  
$$\sigma = \Delta f \cdot \sqrt{2 \ln(1.25/\delta)} / \varepsilon$$
achieves $(\varepsilon, \delta)$-DP. In DP-SGD: $\Delta f = C$ (the clipping threshold).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate gradient clipping
rng = np.random.default_rng(42)
grad_norms = rng.exponential(scale=2.0, size=500)   # simulated raw gradient norms
C = 1.0                                              # max_grad_norm
clipped_norms = np.minimum(grad_norms, C)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(grad_norms, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(C, color='red', lw=2, label=f'max_grad_norm = {C}')
axes[0].set_title('Raw per-sample gradient norms')
axes[0].set_xlabel('‖∇L‖₂')
axes[0].legend()

axes[1].hist(clipped_norms, bins=40, color='coral', edgecolor='white')
axes[1].axvline(C, color='red', lw=2, label=f'max_grad_norm = {C}')
axes[1].set_title('After clipping — sensitivity bounded to C')
axes[1].set_xlabel('‖∇L‖₂  (clipped)')
axes[1].legend()

plt.tight_layout()
plt.show()

pct_clipped = (grad_norms > C).mean() * 100
print(f"Fraction of gradients clipped: {pct_clipped:.1f}%")
print(f"After clipping, max sensitivity = {clipped_norms.max():.4f}  (should equal C = {C})")

In [ ]:
# Noise multiplier vs epsilon (closed-form Gaussian mechanism)
import numpy as np
import matplotlib.pyplot as plt

delta = 1e-5
epsilons = np.linspace(0.5, 12, 200)
# Gaussian mechanism: sigma = sqrt(2*ln(1.25/delta)) / epsilon
sigmas = np.sqrt(2 * np.log(1.25 / delta)) / epsilons

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epsilons, sigmas, lw=2)
for eps, label in [(3, 'ε=3'), (10, 'ε=10')]:
    s = np.sqrt(2 * np.log(1.25 / delta)) / eps
    ax.scatter([eps], [s], zorder=5)
    ax.annotate(f'{label}\nσ≈{s:.2f}', (eps, s), textcoords='offset points', xytext=(8, 4))
ax.set_xlabel('ε')
ax.set_ylabel('Noise multiplier σ (closed-form)')
ax.set_title('Stronger privacy (lower ε)  requires  more noise')
ax.set_ylim(0, 15)
plt.tight_layout()
plt.show()

print("Note: Opacus uses RDP accounting, so actual σ differs slightly from the closed-form values above.")
print("Typical Opacus values for CIFAR-10 (20 epochs, batch=256):")
print("  ε=10  →  σ ≈ 1.2")
print("  ε= 3  →  σ ≈ 3.8")

### Privacy Composition

Each gradient step consumes a slice of the privacy budget. Opacus uses **Rényi Differential Privacy (RDP)** accounting to track the cumulative spend tightly.

Key intuition: to maintain the *same final ε* across more training steps, each step must leak *less* — requiring a **higher noise multiplier** — making optimisation harder.

In [ ]:
# Visualise how epsilon accumulates across training steps (naive linear composition)
steps = np.arange(1, 201)
epsilon_per_step = 0.05   # hypothetical per-step ε

naive_total = steps * epsilon_per_step
# Advanced composition grows roughly as sqrt(k) * ε_per_step for illustration
advanced_total = np.sqrt(steps) * epsilon_per_step * 3  # rough illustration only

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, naive_total, label='Naive composition  (Σ ε_i)', lw=2)
ax.plot(steps, advanced_total, label='Advanced / RDP  (tighter bound)', lw=2, ls='--')
ax.set_xlabel('Training steps')
ax.set_ylabel('Total ε')
ax.set_title('Privacy budget accumulates — RDP tracking stays tighter')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Opacus — Architecture Constraints

| Layer | DP-SGD compatible? | Reason |
|-------|--------------------|--------|
| `Conv2d`, `Linear` | ✅ Yes | Independent per-sample computation |
| `BatchNorm` | ❌ No | Computes statistics *across* the batch — couples sample gradients |
| `GroupNorm`, `LayerNorm`, `InstanceNorm` | ✅ Yes | Normalise *within* each sample |

`ModuleValidator.fix()` auto-replaces `BatchNorm` → `GroupNorm` where possible.

In [ ]:
# Show the CIFAR10CNN architecture used throughout the demo
import torch
import torch.nn as nn

class CIFAR10CNN_Demo(nn.Module):
    """Minimal reproduction of the htb_ai_library CIFAR10CNN.
    BatchNorm is intentionally absent — all layers are Opacus-compatible."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CIFAR10CNN_Demo()
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Demonstrate ModuleValidator — flag & fix BatchNorm
from opacus.validators import ModuleValidator

class ModelWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),   # incompatible with Opacus
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(16 * 32 * 32, 10),
        )
    def forward(self, x):
        return self.net(x)

bad_model = ModelWithBatchNorm()
errors = ModuleValidator.validate(bad_model, strict=False)
print("Errors before fix:", errors)

fixed_model = ModuleValidator.fix(bad_model)
errors_after = ModuleValidator.validate(fixed_model, strict=False)
print("Errors after fix :", errors_after)
print("\nFixed architecture:")
print(fixed_model)

---
## 4. Configuration

In [ ]:
import os
import json
import torch
import torch.optim as optim
from safetensors.torch import save_file
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

from htb_ai_library import (
    set_reproducibility, use_htb_style,
    CIFAR10CNN,
    get_cifar10_loaders,
    train_baseline_sgd,
    train_dp_sgd,
    evaluate_accuracy,
    compute_mia_advantage,
    plot_accuracy_comparison,
    plot_privacy_utility_tradeoff,
)

# ── Hyperparameters ──────────────────────────────────────────────────────────
RANDOM_SEED    = 1337
BATCH_SIZE     = 256
BASELINE_EPOCHS = 20
BASELINE_LR    = 0.1
DP_EPOCHS      = 20
DP_LR          = 0.1
MAX_GRAD_NORM  = 1.0    # L2 clipping threshold
DELTA          = 1e-5   # failure probability — satisfies δ < 1/50000
TARGET_EPSILON_10 = 10.0
TARGET_EPSILON_3  = 3.0

set_reproducibility(RANDOM_SEED)
use_htb_style()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"Seed   : {RANDOM_SEED}")

for d in ('figs', 'output', 'models'):
    os.makedirs(d, exist_ok=True)

---
## 5. Load CIFAR-10

In [ ]:
train_dataset, test_dataset, train_loader, test_loader = get_cifar10_loaders(
    batch_size=BATCH_SIZE, download=True
)

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Batch size       : {BATCH_SIZE}")
print(f"Poisson rate q   : {BATCH_SIZE / len(train_dataset):.5f}  (used by Opacus for privacy amplification)")

In [ ]:
# Quick peek at the data
import matplotlib.pyplot as plt
import numpy as np

CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

images, labels = next(iter(train_loader))
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3,1,1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = images[i] * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(CLASSES[labels[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 sample images', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Baseline Model (No Privacy)

In [ ]:
print("=" * 60)
print("  TRAINING: BASELINE (No DP)")
print("=" * 60)

baseline_model = CIFAR10CNN().to(device)
baseline_model = train_baseline_sgd(
    baseline_model, train_loader, device,
    epochs=BASELINE_EPOCHS, learning_rate=BASELINE_LR
)

In [ ]:
train_acc_baseline = evaluate_accuracy(baseline_model, train_loader, device)
test_acc_baseline  = evaluate_accuracy(baseline_model, test_loader,  device)

print("Baseline model performance:")
print(f"  Training accuracy : {train_acc_baseline:.2f}%")
print(f"  Test accuracy     : {test_acc_baseline:.2f}%")
print(f"  Overfitting gap   : {train_acc_baseline - test_acc_baseline:.2f}%")

torch.save(baseline_model.state_dict(), 'output/baseline_model.pth')

---
## 7. Membership Inference Attack on the Baseline

**Confidence-threshold MIA**:
1. Collect max softmax confidence for all training samples (members) and all test samples (non-members)
2. Balance to 10,000 each (CIFAR-10 has exactly 10,000 test samples)
3. Sweep thresholds; report the threshold that maximises attack accuracy

**Advantage** = attack accuracy − 0.5  (0 = no better than random; 0.5 = perfect)

In [ ]:
print("=" * 60)
print("  MIA: Baseline Model")
print("=" * 60)

mia_acc_baseline, mia_adv_baseline = compute_mia_advantage(
    baseline_model, train_loader, test_loader, device
)

print(f"\nAttack accuracy  : {mia_acc_baseline:.4f}")
print(f"Attack advantage : {mia_adv_baseline:.4f}")
print(f"Random baseline  : 0.5000")
print(f"\nInterpretation: the attacker is {mia_adv_baseline*100:.1f}pp above random guessing.")

In [ ]:
# Visualise confidence gap: members vs non-members
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

@torch.no_grad()
def get_max_confidences(model, loader, device, n_max=10_000):
    model.eval()
    confs = []
    for x, _ in loader:
        x = x.to(device)
        probs = F.softmax(model(x), dim=1)
        confs.extend(probs.max(dim=1).values.cpu().tolist())
        if len(confs) >= n_max:
            break
    return confs[:n_max]

train_confs = get_max_confidences(baseline_model, train_loader, device)
test_confs  = get_max_confidences(baseline_model, test_loader,  device)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(train_confs, bins=60, alpha=0.6, label='Members (train)', density=True)
ax.hist(test_confs,  bins=60, alpha=0.6, label='Non-members (test)', density=True)
ax.set_xlabel('Max softmax confidence')
ax.set_ylabel('Density')
ax.set_title('Confidence gap enables membership inference (Baseline)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. DP-SGD Model — ε = 10 (Modest Privacy)

In [ ]:
print("=" * 60)
print(f"  TRAINING: DP-SGD  (target ε = {TARGET_EPSILON_10})")
print("=" * 60)

# Each DP model needs its own loader — Opacus wraps it for privacy accounting
_, _, train_loader_dp10, test_loader_dp10 = get_cifar10_loaders(
    batch_size=BATCH_SIZE, download=False
)

dp_model_10  = CIFAR10CNN().to(device)
dp_model_10  = ModuleValidator.fix(dp_model_10)
optimizer_10 = optim.SGD(dp_model_10.parameters(), lr=DP_LR, momentum=0.9)

privacy_engine_10 = PrivacyEngine(accountant='rdp')
dp_model_10, optimizer_10, train_loader_dp10 = privacy_engine_10.make_private_with_epsilon(
    module       = dp_model_10,
    optimizer    = optimizer_10,
    data_loader  = train_loader_dp10,
    target_epsilon = TARGET_EPSILON_10,
    target_delta   = DELTA,
    epochs         = DP_EPOCHS,
    max_grad_norm  = MAX_GRAD_NORM,
)

print(f"  Noise multiplier (σ): {optimizer_10.noise_multiplier:.4f}")
print(f"  Max grad norm    (C): {MAX_GRAD_NORM}")

In [ ]:
dp_model_10, final_epsilon_10 = train_dp_sgd(
    dp_model_10, train_loader_dp10, privacy_engine_10, optimizer_10, device
)

print(f"\nFinal privacy guarantee: (ε={final_epsilon_10:.2f}, δ={DELTA})")

In [ ]:
train_acc_dp10 = evaluate_accuracy(dp_model_10, train_loader_dp10, device)
test_acc_dp10  = evaluate_accuracy(dp_model_10, test_loader_dp10,  device)

print(f"DP Model (ε=10) Performance:")
print(f"  Training accuracy : {train_acc_dp10:.2f}%")
print(f"  Test accuracy     : {test_acc_dp10:.2f}%")
print(f"  Overfitting gap   : {train_acc_dp10 - test_acc_dp10:.2f}%")

torch.save(dp_model_10._module.state_dict(), 'output/dp_model_eps10.pth')
save_file(dp_model_10._module.state_dict(), 'models/dp_model.safetensors')
print("\nSaved to models/dp_model.safetensors")

In [ ]:
mia_acc_dp10, mia_adv_dp10 = compute_mia_advantage(
    dp_model_10, train_loader_dp10, test_loader_dp10, device
)

improvement_10 = mia_adv_baseline - mia_adv_dp10
print(f"MIA Results (DP ε=10):")
print(f"  Attack accuracy  : {mia_acc_dp10:.4f}")
print(f"  Attack advantage : {mia_adv_dp10:.4f}")
print(f"\nVs Baseline:")
print(f"  MIA advantage reduction : {improvement_10:+.4f}")
print(f"  Accuracy cost           : {test_acc_baseline - test_acc_dp10:.2f}pp")

---
## 9. DP-SGD Model — ε = 3 (Stronger Privacy)

In [ ]:
print("=" * 60)
print(f"  TRAINING: DP-SGD  (target ε = {TARGET_EPSILON_3})")
print("=" * 60)

_, _, train_loader_dp3, test_loader_dp3 = get_cifar10_loaders(
    batch_size=BATCH_SIZE, download=False
)

dp_model_3  = CIFAR10CNN().to(device)
dp_model_3  = ModuleValidator.fix(dp_model_3)
optimizer_3 = optim.SGD(dp_model_3.parameters(), lr=DP_LR, momentum=0.9)

privacy_engine_3 = PrivacyEngine(accountant='rdp')
dp_model_3, optimizer_3, train_loader_dp3 = privacy_engine_3.make_private_with_epsilon(
    module       = dp_model_3,
    optimizer    = optimizer_3,
    data_loader  = train_loader_dp3,
    target_epsilon = TARGET_EPSILON_3,
    target_delta   = DELTA,
    epochs         = DP_EPOCHS,
    max_grad_norm  = MAX_GRAD_NORM,
)

print(f"  Noise multiplier (σ): {optimizer_3.noise_multiplier:.4f}")
print(f"  Max grad norm    (C): {MAX_GRAD_NORM}")

In [ ]:
dp_model_3, final_epsilon_3 = train_dp_sgd(
    dp_model_3, train_loader_dp3, privacy_engine_3, optimizer_3, device
)

print(f"\nFinal privacy guarantee: (ε={final_epsilon_3:.2f}, δ={DELTA})")

In [ ]:
train_acc_dp3 = evaluate_accuracy(dp_model_3, train_loader_dp3, device)
test_acc_dp3  = evaluate_accuracy(dp_model_3, test_loader_dp3,  device)

print(f"DP Model (ε=3) Performance:")
print(f"  Training accuracy : {train_acc_dp3:.2f}%")
print(f"  Test accuracy     : {test_acc_dp3:.2f}%")
print(f"  Overfitting gap   : {train_acc_dp3 - test_acc_dp3:.2f}%")

torch.save(dp_model_3._module.state_dict(), 'output/dp_model_eps3.pth')

mia_acc_dp3, mia_adv_dp3 = compute_mia_advantage(
    dp_model_3, train_loader_dp3, test_loader_dp3, device
)

improvement_3 = mia_adv_baseline - mia_adv_dp3
print(f"\nMIA Results (DP ε=3):")
print(f"  Attack accuracy  : {mia_acc_dp3:.4f}")
print(f"  Attack advantage : {mia_adv_dp3:.4f}")
print(f"\nVs Baseline:")
print(f"  MIA advantage reduction : {improvement_3:+.4f}")
print(f"  Accuracy cost           : {test_acc_baseline - test_acc_dp3:.2f}pp")

---
## 10. Comparison Table & Visualisations

In [ ]:
def print_comparison_table(test_acc_baseline, test_acc_dp10, test_acc_dp3,
                           mia_adv_baseline,  mia_adv_dp10,  mia_adv_dp3,
                           final_epsilon_10,  final_epsilon_3):
    print("\n" + "=" * 65)
    print("  COMPARISON TABLE")
    print("=" * 65)
    print(f"{'Model':<20} {'Test Acc':>10} {'MIA Adv':>10} {'Epsilon':>10}")
    print("-" * 55)
    print(f"{'Baseline':<20} {test_acc_baseline:>9.2f}% {mia_adv_baseline:>10.4f} {'∞':>10}")
    print(f"{'DP (ε=10)':<20} {test_acc_dp10:>9.2f}% {mia_adv_dp10:>10.4f} {final_epsilon_10:>10.2f}")
    print(f"{'DP (ε=3)':<20} {test_acc_dp3:>9.2f}% {mia_adv_dp3:>10.4f} {final_epsilon_3:>10.2f}")

print_comparison_table(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    mia_adv_baseline,  mia_adv_dp10,  mia_adv_dp3,
    final_epsilon_10,  final_epsilon_3
)

In [ ]:
plot_accuracy_comparison(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    save_path='figs/dp_sgd_accuracy_comparison.png'
)
print("Saved → figs/dp_sgd_accuracy_comparison.png")

In [ ]:
plot_privacy_utility_tradeoff(
    test_acc_baseline, test_acc_dp10, test_acc_dp3,
    mia_adv_baseline,  mia_adv_dp10,  mia_adv_dp3,
    final_epsilon_10,  final_epsilon_3,
    save_path='figs/dp_sgd_privacy_utility_tradeoff.png'
)
print("Saved → figs/dp_sgd_privacy_utility_tradeoff.png")

In [ ]:
# Manual tradeoff plot — accuracy AND MIA advantage on the same axes
import matplotlib.pyplot as plt

epsilons_plot = [3, 10, float('inf')]   # x-axis: privacy strength (left=strong)
accs  = [test_acc_dp3, test_acc_dp10, test_acc_baseline]
mia   = [mia_adv_dp3,  mia_adv_dp10,  mia_adv_baseline]
xlabels = ['ε=3\n(strong)', 'ε=10\n(moderate)', 'Baseline\n(no DP)']

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

x = range(len(xlabels))
ax1.plot(x, accs, 'o-', color='steelblue',  lw=2, label='Test accuracy (%)')
ax2.plot(x, mia,  's--', color='tomato',    lw=2, label='MIA advantage')

ax1.set_xticks(x)
ax1.set_xticklabels(xlabels)
ax1.set_ylabel('Test accuracy (%)', color='steelblue')
ax2.set_ylabel('MIA advantage',     color='tomato')
ax1.set_title('Privacy-Utility Tradeoff (stronger privacy → left)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.savefig('figs/manual_tradeoff.png', dpi=150)
plt.show()

---
## 11. Save All Results

In [ ]:
results = {
    'baseline': {
        'test_accuracy' : float(test_acc_baseline),
        'mia_advantage' : float(mia_adv_baseline),
    },
    'dp_eps10': {
        'test_accuracy' : float(test_acc_dp10),
        'mia_advantage' : float(mia_adv_dp10),
        'epsilon'       : float(final_epsilon_10),
    },
    'dp_eps3': {
        'test_accuracy' : float(test_acc_dp3),
        'mia_advantage' : float(mia_adv_dp3),
        'epsilon'       : float(final_epsilon_3),
    },
}

with open('output/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to output/results.json")
print(json.dumps(results, indent=2))

---
## 12. Summary & Key Takeaways

| Model | Test Acc | MIA Adv | Overfitting Gap |
|-------|----------|---------|----------------|
| Baseline (no DP) | ~67% | ~0.019 | ~10% |
| DP ε=10 | ~58% | ~0.008 | ~3% |
| DP ε=3  | ~53% | ~0.004 | ~1% |

### What the numbers tell us

1. **Privacy protection scales with noise.** Each halving of ε roughly halves the MIA advantage.
2. **Overfitting gap collapses under DP.** The baseline memorises training patterns (10% gap); ε=3 shows almost none (1% gap).
3. **Utility cost is bounded but real.** ε=3 costs ~14pp accuracy vs baseline — the model is still useful.
4. **Theoretical guarantee vs empirical measurement** are complementary:
   - DP provides a *worst-case upper bound* (holds against any future attack).
   - Empirical MIA gives a *current lower bound* (measures a specific known attack today).

### Architecture rule of thumb
- Replace `BatchNorm` with `GroupNorm` before attaching Opacus.
- Use `ModuleValidator.fix()` to catch anything you missed.

### When *not* to use DP-SGD
- Very small datasets — noise drowns the signal.
- ε < 1 without public pre-training — accuracy usually drops below useful levels.
- Threat model requires protecting aggregate statistics, not individual samples.

In [ ]:
print("=" * 60)
print("  DEMONSTRATION COMPLETE")
print("=" * 60)
print(f"  Baseline  MIA advantage : {mia_adv_baseline:.4f}")
print(f"  DP ε=10   MIA advantage : {mia_adv_dp10:.4f}  ({improvement_10:+.4f})")
print(f"  DP ε=3    MIA advantage : {mia_adv_dp3:.4f}  ({improvement_3:+.4f})")
print("=" * 60)